# 🤖 Agentic AI with Bigdata.com Search API

This notebook demonstrates how your AI agents can interact with **Bigdata.com Search API** to access real-time market intelligence, news, filings, and earnings transcripts—combined with your internal data sources.

## What This Demonstrates

This notebook delivers a cited, multi-source answer in one flow—combining your internal portfolios and research with **Bigdata.com** for the external side: **Knowledge Graph** resolves companies to entity IDs for precise filtering, and the **Search API** delivers news, filings, and transcripts with sentiment. Your agent leans on Bigdata.com for real-time market intelligence and entity-scoped search, then weaves in internal data for a single, traceable response with inline source links.

**Bigdata.com Integration Patterns:**
- **Knowledge Graph API** → Resolve company names to entity IDs for precise filtering
- **Search API (smart mode by default)** → Send a natural-language question and let the API resolve entities, time ranges, document types, and ranking automatically; drop into fast mode with explicit filters when you need deterministic precision
- **Tool-based Architecture** → Wrap APIs as callable tools for any agentic framework

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine external market data with proprietary insights

**Framework Flexibility:**
> This demo uses **LangChain** and **LangSmith** for observability, but the integration pattern applies to any agentic framework: **CrewAI**, **AutoGen**, **Google A2A**, or custom implementations. The key is wrapping Bigdata.com APIs as tools your agents can call.

---

## Architecture

![Agent to Bigdata APIs ](./static/agent-search.png)

**Key Points:**
- **Single agent interface** for internal DB, vector store, and Bigdata.com Search/Knowledge Graph
- **Production-ready** tooling: retry, logging, KG entity cache
- **Observability** via LangSmith for tool selection and latency
- **Flexibility**: Same pattern works with CrewAI, AutoGen, or custom frameworks—wrap APIs as tools

---

## 1️⃣ Install Dependencies

Install from the project root before running this notebook:

```bash
uv sync
```

In [4]:
# Dependencies: install from project root with uv sync (see README)

## 2️⃣ Import Libraries

Import core utilities and display helpers from `langgraph_core`.

In [5]:
import os
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML

# LangChain
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI

# Local utilities - environment, data sources, and display helpers
import sys
sys.path.append('.')
from langgraph_core import (
    # Environment & Data Setup
    setup_environment,
    create_financial_database,
    create_vector_store,
    
    # Tool Providers
    get_bigdata_tools,
    get_database_tools,
    get_vectorstore_tools,
    
    # Display Utilities
    display_query,
    display_response,
    display_tools_used,
    display_citations
)

# Load environment variables
load_dotenv()

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [6]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project='bigdata-agent-demo',
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")
print("\n🔗 View traces at: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)

🔗 View traces at: https://smith.langchain.com


## 4️⃣ Load Local Tools

Load tools that interact with local data sources.

In [7]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for tool in local_db_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for tool in local_vector_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transaction...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity...


## 5️⃣ Load Bigdata.com Tools

Load external tools for market intelligence:

- **Knowledge Graph API** - Entity lookup and company identification
- **Search API** - News, filings, transcripts, and research with sentiment (smart mode by default, fast mode for precision filters)
- **Other** - can be added based on need

In [8]:
# Get Bigdata.com tools
bigdata_tools = get_bigdata_tools()
print(f"✅ Loaded {len(bigdata_tools)} Bigdata.com tools:")
for tool in bigdata_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 Bigdata.com tools:
   - bigdata_lookup_company: Look up a company's Bigdata entity ID using the Knowledge Gr...
   - bigdata_search: Search Bigdata.com for news, filings, transcripts, and resea...


## 6️⃣ Combine All Tools

Merge tools from all sources for the agent.

In [9]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata.com tools: {len(bigdata_tools)}")


✅ Total tools available: 5
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata.com tools: 2


## 7️⃣ Create LangChain Agent

Create an agent with all tools configured.

**ReAct Pattern:**
- **Reasoning**: Plans which tools to use based on query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

**Note:** Update it based on need.

In [10]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com APIs):**
1. `bigdata_lookup_company` - Look up company entity IDs from the Knowledge Graph
2. `bigdata_search` - Search news, filings, transcripts, and research (defaults to smart mode)

**Internal Data (Company Systems):**
3. `internal_query_database` - Execute SQL queries on portfolio/transaction database
4. `internal_portfolio_summary` - Get portfolio holdings and performance summary
5. `internal_search_research` - Search internal investment research documents

**How to use `bigdata_search`:**
- DEFAULT to smart mode. Pass the user's question (or a lightly clarified version) as a natural-language sentence in `query`, including entity names, the time period, and any content hint (e.g. "news", "earnings call"). Smart mode resolves entities, temporal ranges, document types, and ranking for you and expands sub-queries for coverage. Use it for macro/FX topics (e.g. "market outlook on the Brazilian Real") that do not map to a single company.
- Keep semantic concepts, themes, and qualitative drivers in `query`. Do NOT stuff tickers or document-type keywords into the query when a filter can express them.
- Use FAST mode only when you need deterministic precision. First resolve companies with `bigdata_lookup_company`, then choose the right entity filter:
  - `reporting_entity_ids` -> documents AUTHORED/filed by the company (earnings calls, transcripts, 10-K/10-Q/8-K, investor presentations).
  - `entity_ids` -> documents that MENTION or discuss the company (news, third-party commentary).
  Avoid using both for the same company. Add `category`/`document_type`, a tight `days_back` (30 for "recent/latest", 90 default), and a moderate `rerank_threshold` to cut boilerplate.
- Iterative fallback: start precision-first; if too little is returned, relax constraints (widen `days_back`, lower `rerank_threshold`, broaden `document_type`, or switch `reporting_entity_ids` -> `entity_ids`).

Guidelines:
- For portfolio questions, use `internal_portfolio_summary` or `internal_query_database` with SQL
- Combine external market data with internal holdings/research for comprehensive analysis

**Citation format:** Use inline citations with the **source name as the link text** (not the raw URL). Format as markdown: [Source Name](url) or [1](url), [2](url) so the reader sees a clickable source name. Do not paste full URLs in the body.

**Do not add a separate "Sources" or "References" block at the end** when you have already used inline citations. Inline citations are sufficient. For internal data: Briefly mention "From internal database" or "According to internal research."
**Do not offer suggestions for follow up questions**

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

# Initialize LLM
model = "gpt-5" 
llm = ChatOpenAI(
    model=model,
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create agent using langchain.agents.create_agent
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: {model}")
print(f"   System prompt configured")
print("\n🔍 Agent Tool Selection:")
print("   • Portfolio/holdings → internal_query_database")
print("   • Internal research → internal_search_research")
print("   • Company lookup → bigdata_lookup_company")
print("   • Market intelligence → bigdata_search (smart mode by default)")

✅ Agent created with 5 tools
   Model: gpt-5
   System prompt configured

🔍 Agent Tool Selection:
   • Portfolio/holdings → internal_query_database
   • Internal research → internal_search_research
   • Company lookup → bigdata_lookup_company
   • Market intelligence → bigdata_search (smart mode by default)


## 8️⃣ System Prompt

The agent uses the `SYSTEM_PROMPT` defined above. Key elements:

**External Tools (Bigdata.com):**
- `bigdata_lookup_company` - Get entity IDs (used only for fast-mode precision filters)
- `bigdata_search` - Search news, filings, transcripts, and research (smart mode by default)

**Internal Tools:**
- `internal_query_database` - SQL queries on portfolios
- `internal_portfolio_summary` - Holdings & performance
- `internal_search_research` - Semantic search on research docs

**Search routing:**
- **Smart mode (default):** send the natural-language question; the API resolves entities, time ranges, document types, and ranking, and runs sub-queries for coverage. Best for most questions and for macro/FX topics that are not a single company.
- **Fast mode (precision):** resolve entities first, then choose `reporting_entity_ids` (company-authored: filings, earnings calls) vs `entity_ids` (mentions/third-party), add `category`/`document_type`, a tight time window, and a moderate `rerank_threshold`. Relax these iteratively if recall is too low.

**Citation:** Inline citations with **source name as hyperlink** — format as [Source Name](url). Do not add a separate "Sources" block; citations are rendered inline.

---

## 🔎 Search Best Practices: Smart vs. Fast Mode

`bigdata_search` defaults to **smart mode**, which is the recommended path for agents. You only formulate `query.text`; the API interprets the natural-language request — extracting entities, temporal expressions, and intent — and converts it into a structured search (entity filters, content and temporal constraints), running sub-queries for broader coverage. This increases precision and avoids costly trial-and-error from an agent that has not been trained on search generation.

**Fast mode** gives full control over filters for precise, deterministic, low-latency results, but has no query understanding. Use it when the router has already resolved entities and wants explicit control.

| Parameter | Fast mode | Smart mode |
|---|---|---|
| Text | ✅ | ✅ |
| Temporal / source filter | ✅ Manual | ✅ Automated |
| Document type / entity / reporting entity | ✅ Manual | ✅ Fully automated |
| Keyword / sentiment / ranking | ✅ Manual | ✅ Fully automated |

**Entity routing (fast mode):**
- `entity_ids` → documents that **mention** the company (news, third-party commentary).
- `reporting_entity_ids` → documents **authored/filed by** the company (earnings calls, transcripts, 10-K/10-Q/8-K, presentations).

**Precision vs. recall:** start precision-first (smart mode, or fast mode with tight filters and a moderate `rerank_threshold`). If too little is returned, relax constraints — widen `days_back`, lower `rerank_threshold`, broaden `document_type`, or switch `reporting_entity_ids` → `entity_ids`.

**Tip:** set `include_audit=True` to inspect the resolved queries smart mode executed, then replicate them in fast mode if you need deterministic control.



---

## 9️⃣ Example Queries

Run queries that combine data from multiple sources.

### Query 1: Multi-Source NVIDIA Analysis

Combines internal holdings, research, and external news.

In [17]:
query = """I need a comprehensive analysis of NVIDIA:
1. What's our current position in NVIDIA across all portfolios?
2. What does our internal research say about NVIDIA's investment thesis?
3. What's the latest news about NVIDIA from market sources?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a consolidated view on NVIDIA:

1) Our current NVDA position across all portfolios (from internal database)
- PF002 (AI & Semiconductor Focus; AUM $15.0M): 12,000 shares at current internal price $875.50; market value $10.506M; unrealized P&L $5.106M; portfolio weight ~70.0%.
- PF003 (Diversified Tech Leaders; AUM $50.0M): 8,000 shares at $875.50; market value $7.004M; unrealized P&L $2.844M; portfolio weight ~14.0%.
- PF001 (US Large Cap Growth; AUM $30.0M): No current NVDA holding in the holdings table.
- Aggregate across PF001–PF003: 20,000 shares; market value $17.51M; unrealized P&L $7.95M. This is ~18.4% of combined AUM for these portfolios.

2) What our internal research says (summary)
According to internal research:
- Core thesis: NVIDIA remains the dominant AI accelerator platform, with data center growth led by H100/H200 and the next-gen Blackwell architecture (B100/B200) expected to extend performance leadership. The CUDA software ecosystem (4M+ developers) creates high switching costs and a durable moat. Inference is emerging as a second, very large leg of growth as enterprises deploy models at scale.
- Key growth drivers:
  - Datacenter AI accelerators and systems (DGX/GB200), plus networking (InfiniBand/Ethernet) and NVLink fabric.
  - Software stack (CUDA, cuDNN, TensorRT, enterprise software) enabling performance and lock-in.
  - Inference TAM expansion (internal estimate cited: ~$150B by 2027).
- Valuation and stance: Prior internal thesis update (Dec 2024) carried a Strong Buy, $950 PT on ~25x FY26E EPS; strategy memos in early 2025 recommended increasing NVDA allocation given AI demand outstripping supply. Risk memos flagged concentration/valuation risk and geopolitics.
- Key risks: Export restrictions to China (internal estimate 20–25% revenue at risk), competitive pressure from AMD (MI300/MI325 and ROCm improvement) and custom silicon, supply constraints, and potential AI ROI/monetization shortfalls.
- Catalysts to watch: Blackwell ramp, enterprise AI adoption/inference monetization, networking attach (InfiniBand/Ethernet), and any broadening of the software monetization model. (From internal research)

3) Latest NVIDIA news and market developments (last ~30 days)
- Roadmap/disruption chatter and rebuttal: Reports circulated that NVIDIA’s next-gen rack-scale “Kyber NVL144” system could be delayed by more than a year; NVIDIA said “our roadmap is intact,” denying that core progress has slipped. Shares bounced on the rebuttal [Yahoo! Finance](https://finance.yahoo.com/markets/stocks/articles/nvda-stock-climbs-over-1-173553076.html), [Yahoo! Finance](https://uk.finance.yahoo.com/news/nvidia-denies-report-its-next-generation-ai-server-faces-delays-says-roadmap-is-intact-183310296.html).
- Blackwell ramp: Commentary notes NVIDIA is now shipping its highly anticipated Blackwell GPUs, supporting the view that the product cycle continues to progress despite volatility [Nasdaq](https://www.nasdaq.com/articles/3-no-brainer-stocks-buy-latest-sell).
- Customer deployments/partners: Vultr selected HPE and NVIDIA for next-gen AI infrastructure to scale an AI cloud platform—evidence of continued demand outside the hyperscaler “big 5” [Benzinga](https://www.benzinga.com/node/53252922?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
- Competitive backdrop: Several pieces highlight AMD narrowing the gap in AI accelerators and hyperscalers seeking diversification, though NVIDIA’s CUDA/software stack and installed base remain key advantages [AOL.com](https://www.aol.com/articles/forget-nvidia-amd-might-close-175526000.html), [Nasdaq](https://www.nasdaq.com/articles/forget-nvidia-buy-these-4-ai-hardware-stocks-bigger-gains).
- Market sentiment/volatility: Recent sell-off headlines emphasize valuation sensitivity and the importance of sustained AI capex and earnings delivery; some commentary points to growth re-acceleration in 2027 aligned with NVIDIA’s platform cadence [The Globe And Mail](https://www.theglobeandmail.com/investing/markets/markets-news/Motley%20Fool/3142209/nvidia-stock-is-below-200-per-share-again-here-s-when-it-could-hit-300/), [FinanceFeeds](https://financefeeds.com/nvidia-nvda-price-prediction-197-300-bull-150-bear/).

Implications for our position
- The “roadmap intact” response and signs of Blackwell shipments support the internal thesis that NVIDIA remains on schedule operationally, even as competitive intensity rises and headlines can drive volatility.
- Demand indicators (new deployments like Vultr/HPE) are consistent with continued AI infrastructure buildout breadth.
- Risks to monitor closely: further regulatory/export-control developments, competitive wins at scale (e.g., MI300/MI325 share gains), and potential digestion periods in AI capex that could pressure multiples.

If you’d like, I can refresh the positions versus live prices and pull any earnings/transcript highlights as they post.

### Query 2: Portfolio Risk Assessment

Analyzes portfolio holdings with internal research and external news.

In [12]:
query = """Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. Are there any recent news events affecting these holdings?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise update on PF002 (AI & Semiconductor Focus).

1) Current holdings and performance (From internal database)
- Portfolio MV: $15.99m across 5 positions; total unrealized PnL: +$7.77m
- Position detail (weight | unrealized return vs cost):
  - NVDA — 12,000 sh @ $875.50; MV $10.51m (65.7%); PnL +$5.11m; +94.6%
  - AVGO — 1,500 sh @ $1,425.00; MV $2.14m (13.4%); PnL +$0.86m; +67.7%
  - PLTR — 25,000 sh @ $65.25; MV $1.63m (10.2%); PnL +$1.17m; +252.5%
  - AMD — 8,000 sh @ $145.25; MV $1.16m (7.3%); PnL +$0.40m; +52.9%
  - TSM — 3,000 sh @ $185.75; MV $0.56m (3.5%); PnL +$0.23m; +69.0%
- Concentration: NVDA is 65.7% of portfolio MV. Holdings count: 5.

2) Risks highlighted by our internal research (According to internal research)
- Portfolio-level
  - Valuation risk (High): AI leaders carry premium multiples; multiples could compress if AI monetization underwhelms (Jan-2025 Technology Sector Risk Assessment).
  - Regulatory/antitrust (Med-High): Heightened U.S./EU scrutiny across large-cap tech ecosystems (Jan-2025 Risk Assessment).
  - China exposure/export controls (Med): NVDA called out with 20–25% revenue at risk from export restrictions (Jan-2025 Risk Assessment).
  - AI cycle risk (Med): Hyperscaler/data center capex may be front-loaded; ROI on enterprise AI projects still proving out (Jan-2025 Risk Assessment).
- Company-specific
  - NVIDIA: Export restrictions to China, supply constraints, and competitive pressure (AMD/custom silicon) noted as key risks (Dec-2024 NVDA thesis update).
  - AMD: Execution risk vs. NVIDIA’s CUDA ecosystem; ROCm/software maturity gap; valuation premium tied to AI optionality (Dec-2024 AMD thesis).
  - PLTR: Benefiting from accelerating gov’t/commercial AI demand, but sensitive to federal budget/policy cycles and high-growth valuation (Jan-2025 strategy/risk notes).
  - Note: These are the principal themes surfaced in our latest internal memos and theses; they are not exhaustive.

3) Recent news/events impacting holdings (last ~30 days)
- NVIDIA (NVDA)
  - Reports point to Blackwell GPU shipments beginning and continued robust AI data-center demand momentum [Nasdaq](https://www.nasdaq.com/articles/3-no-brainer-stocks-buy-latest-sell).
- Broadcom (AVGO)
  - Extended Apple custom-chip supply agreement through 2031, reinforcing long-term custom silicon revenue visibility [Yahoo! Finance](https://finance.yahoo.com/technology/articles/broadcom-extends-apple-chip-supply-145100361.html).
  - Broader AI/custom ASIC narrative: OpenAI and Broadcom highlighted a custom inference chip (“Jalapeño”), underscoring AVGO’s role in AI silicon design/services [Benzinga](https://www.benzinga.com/node/60083543?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
- Palantir (PLTR)
  - Continued traction for AIP commercialization (boot camps cited as key driver) and positive analyst updates highlighting expanding moat and guidance raises [Nasdaq](https://www.nasdaq.com/articles/5-top-ranked-growth-stocks-buy-july-after-mixed-june), [Benzinga](https://www.benzinga.com/node/60244503?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Seeking Alpha](https://seekingalpha.com/article/4920073-palantir-is-turning-ai-into-institutional-infrastructure).
  - Political backdrop: op-ed attention on defense-software providers and large U.S. government awards context [Washington Post](https://www.washingtonpost.com/opinions/2026/07/06/democrats-should-respect-military-software-providers/).
- AMD (AMD)
  - Customer signal: a self-driving startup disclosed ~10% of its AI training now runs on AMD GPUs, pointing to incremental accelerator adoption [Benzinga](https://www.benzinga.com/node/60275051?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Commentary on stretched valuation and supply tightness at advanced nodes/packaging potentially limiting upside near term [Zacks.com](https://www.zacks.com/stock/news/2948014/amd-s-valuation-is-stretched-at-54-08x-p-e-buy-sell-or-hold-the-stock?cid=CS-ZC-FT-analyst_blog%7Cmost_popular_stocks-2948014), [Nasdaq](https://www.nasdaq.com/articles/amds-valuation-stretched-5408x-p-e-buy-sell-or-hold-stock).
- Taiwan Semiconductor (TSM)
  - Reports of potential 3nm price hikes (up to ~15%) amid persistent tightness as AI demand outstrips supply [Benzinga](https://www.benzinga.com/node/53160816?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Upcoming Q2 results on July 16; sell-side notes point to strong AI-driven mix and margin support [Yahoo! Finance](https://finance.yahoo.com/markets/stocks/articles/dear-taiwan-semi-stock-fans-140002938.html), [Auerbach Grayson & Co.](https://research.bluematrix.com/docs/pdf/5e8bc6d6-a8bf-4e0d-8033-f6d2b1ec0818/8094b7c8-1a8f-47c2-b9ed-e42aa9dff21c).

If you want, I can break the news into a one-page dashboard by ticker with links and a quick read on potential portfolio impact.

### Query 3: News Sentiment Analysis

Focuses on external market intelligence with sentiment.

In [13]:
query = """What's the market sentiment around Apple in the last 30 days?
Look for news about:
1. Product launches and innovations
2. Financial performance
3. Regulatory issues

Also check if we hold Apple in any portfolios."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise read on Apple’s market sentiment over the last 30 days (through Jul 6, 2026):

Overall sentiment: Moderately positive, supported by AI/feature momentum and resilient margins, with overhangs from regulatory scrutiny and litigation.

1) Product launches and innovations
- WWDC26: Apple introduced Siri AI and next‑gen Apple Intelligence across iPhone, iPad, Mac and Watch. Coverage was broadly constructive on deeper integration and on‑device privacy positioning [Financial Times](https://www.ft.com/content/28205edf-baeb-4b97-a89d-e9f4838ff6ff), [Benzinga](https://www.benzinga.com/node/53069725?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
- Market reaction: Shares were volatile around WWDC—brief strength followed by profit‑taking as investors digested the announcements and timelines, reflecting a “show‑me” stance on near‑term monetization [Benzinga](https://www.benzinga.com/node/53071511?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

2) Financial performance
- Margins/services remain a key bull pillar: Commentary in recent research highlights strong gross margins (~48% in the March quarter cited) and steady services growth, underpinning the stock despite hardware variability [Yahoo! Finance](https://finance.yahoo.com/markets/stocks/articles/apples-rally-pricing-power-ai-155802427.html).
- Capital returns supportive: Ongoing buyback narrative (including a $100B authorization discussed in recent research) continues to frame Apple as a defensive compounder amid AI investment cycles [CrispIdea](https://research.bluematrix.com/docs/pdf/4615d295-907a-4d19-9570-d07b6a3f1227/c194c88a-8a32-48e0-899d-d57fd783bc27).

Net financial sentiment: Positive—pricing power and services mix offset some concerns about device demand variability and component cost pressures.

3) Regulatory and legal issues
- UK app‑store rules: The CMA proposed changes that could force Apple and Google to loosen payment and distribution restrictions—incrementally negative for App Store economics if implemented [Financial Times](https://www.ft.com/content/2c82a371-e97a-4167-98cb-3d32d513f707).
- China: A group of Chinese developers called for an antitrust probe into App Store fees—adds regional regulatory uncertainty [Anadolu Agency](https://www.aa.com.tr/en/asia-pacific/chinese-developers-seek-antitrust-probe-into-apple-over-app-store-fees/3975167).
- Litigation: Apple is contesting a roughly $500m patent ruling at the UK Supreme Court—headline risk persists while appeals proceed [Financial Times](https://www.ft.com/content/d6a0103a-7fca-4e66-b92f-45f6ff3de059).
- EU engagement on AI/assistant interoperability: Apple held talks with EU tech officials regarding Siri AI and potential compliance paths, signaling ongoing scrutiny as AI features roll out [Financial Times](https://www.ft.com/content/807d25c3-f4ac-4402-b815-3aa91018237d).
- Privacy scrutiny: Reports questioned a “Hide My Email” behavior—limited financial impact, but keeps privacy policies in focus [Indian Express](https://indianexpress.com/article/technology/tech-news-technology/apples-privacy-features-face-scrutiny-over-alleged-hide-my-email-flaw-10768002/).

Net regulatory sentiment: Negative bias—heightened antitrust and litigation risks remain an overhang, particularly in the UK/EU and China.

Do we hold Apple?
- Yes. From internal database:
  - PF001 (US Large Cap Growth): 15,000 shares of AAPL; market value ~$2,778,750; unrealized P&L ~$641,250.
  - PF003 (Diversified Tech Leaders): 25,000 shares of AAPL; market value ~$4,631,250; unrealized P&L ~$756,250.
  - Combined: 40,000 shares; market value ~$7,410,000; unrealized P&L ~$1,397,500.

### Query 4: Portfolio Strategy Review

Combines internal strategy memos with current market conditions.

In [14]:
query = """Review our Q1 2025 portfolio strategy:
1. What allocation changes did our internal research recommend?
2. What are our top holdings by market value?
3. Any recent news that might affect our strategy?

Provide a summary of whether we should stay the course or adjust."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise review of our Q1 2025 portfolio strategy and what to do now.

1) Allocation changes our internal research recommended (Q1 2025)
- Increase: NVDA +3%, META +2%, PLTR +1
- Maintain: MSFT, AAPL
- Reduce: AMD −1%, CRM −1%
- Risk posture: Maintain 5–10% cash and consider protective put spreads on QQQ for tail-risk hedging; monitor valuation, regulatory, and China-exposure risks
From internal research (strategy memo Jan 5, 2025; risk assessment Jan 10, 2025).

2) Top holdings by market value (current)
- NVDA — $17.51M
- MSFT — $9.56M
- AAPL — $7.41M
- CRM — $3.26M
- META — $2.63M
- AVGO — $2.14M
- ORCL — $2.10M
- PLTR — $1.63M
- AMZN — $1.35M
- AMD — $1.16M
From internal database.

3) Recent news that could affect the strategy
- AI compute demand and NVDA momentum: NVDA’s latest results showed continued outsized growth in Compute & Networking, underpinned by AI demand (earnings call summary) [Quartr Reports](https://files.quartr.com/reports/d6460aba65b50da0ccc3599962e9e981-2026-05-20-20-36-33.pdf?ref=UmF2ZW5QYWNr).
- China/export controls risk: Prices for Nvidia’s banned AI chips have surged in China’s gray market, highlighting ongoing constraints and compliance risk for China-exposed sales [Financial Times](https://www.ft.com/content/57fcd3ce-464f-4dc2-8ea2-5712d4972c69). China also expanded its export control list, keeping geopolitical supply-chain risks in focus [Al Jazeera](https://www.aljazeera.com/news/2026/6/22/china-adds-10-us-firms-including-rare-earth-miner-to-export-control-list).
- Hyperscaler capex backdrop: Investor scrutiny is rising, but hyperscaler AI/data center capex remains elevated—constructive for AI infra suppliers and cloud platforms we own [Financial Times](https://www.ft.com/content/c9d562d6-1448-4a40-8198-0cb1f0dc2032).
- CRM product adoption: Salesforce’s 10-Q cited sustained growth in Agentforce and Data 360, but we continue to monitor monetization vs. expectations and valuation sensitivity [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1108524/000110852426000127/crm-20260430.htm).
- PLTR execution: Palantir reported strong EPS growth and raised guidance on robust AI software demand—supportive of our incremental add [Nasdaq](https://www.nasdaq.com/articles/palantir-technologies-stock-could-soar-55-1-year-according-wall-street-should-you-buy-it).

Bottom-line recommendation
- Stay the course with our Q1 2025 tilts, with discipline on risk:
  - Overweights to NVDA, META, and PLTR remain supported by ongoing AI demand, positive execution signals, and a strong AI infra/capex backdrop (citations above).
  - Maintain MSFT and AAPL core positions; no change to thesis.
  - Keep reduced exposure to AMD and CRM given valuation/execution sensitivity; recent disclosures don’t warrant reversing our modest underweights yet.
  - Preserve 5–10% cash and maintain/refresh downside hedges (e.g., QQQ put spreads) given regulatory and China-related tail risks, and the potential for valuation volatility.
From internal research; supported by recent external developments cited above.

### Query 5: Macro / FX Outlook (Smart Mode)

A natural-language macro question that does not map to a single company entity. Smart mode interprets intent, resolves the relevant concepts and time frame, and expands sub-queries for coverage — no manual entity resolution or filter construction required. This is exactly the case where feeding a raw or LLM-expanded prompt into fast mode returns noisy, low-relevance chunks (boilerplate, tables of contents, unrelated filings).

In [15]:
query = "What is the market outlook on the Brazilian Real?"

# Display query
display_query(query)

# Run agent (async invocation) — the agent routes this to bigdata_search in smart mode
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Short answer: Near-term choppy but cushioned by high carry; medium-term mildly constructive if the dollar softens and fiscal risks are contained.

What’s driving BRL now
- Policy/carry: COPOM cut the Selic 25 bp to 14.25% in mid‑June (third straight 25 bp cut), characterizing the cycle as cautious amid stickier inflation—so Brazil still offers one of the highest real/carry profiles in EM, which helps cushion selloffs and can attract flows when volatility is low [US News & World Report](https://money.usnews.com/investing/news/articles/2026-06-17/brazil-central-bank-cuts-rates-for-third-straight-meeting-by-25-bps), [Morningstar](https://www.morningstar.com/news/dow-jones/202606178600/brazils-central-bank-keeps-cutting-despite-hot-prices-update). Several houses also note carry remains a key support for BRL performance versus USD when global risk is stable [LBBW FI](https://research.bluematrix.com/docs/pdf/982dd882-1f71-4da4-ae1d-4c275eeab710/aeab69fe-9ba0-48fa-8e92-2a4cb41e7aff), [Natixis – FX Carry](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/8dc7c8a9-f3a2-4efb-862a-631fd60eb88).
- Inflation: Headline re‑accelerated to about 4.7% y/y in May, with projections drifting above target; COPOM flagged the inflation backdrop as a constraint on easing speed, which supports the carry but also caps risk appetite when prints surprise to the upside [Morningstar](https://www.morningstar.com/news/dow-jones/202606178600/brazils-central-bank-keeps-cutting-despite-hot-prices-update), [US News & World Report](https://money.usnews.com/investing/news/articles/2026-06-17/brazil-central-bank-cuts-rates-for-third-straight-meeting-by-25-bps).
- Fiscal/politics: Markets remain sensitive to fiscal headlines and debt dynamics; analysts highlight high public debt and the need for credible consolidation, while pre‑election political noise has periodically weighed on the currency [The Economist Intelligence Unit](https://cdn.ravenpack.com/uuid/EEE66DC07D96568F7C84741B1F04D5F2/filename/eiu_business-environment-and-the-long-term-forecast_2026-06-25_BR.pdf), [KB Securities](https://research.bluematrix.com/docs/pdf/0912582e-8984-4d23-8cc6-a1e1bb36300d/beca76dc-f2d7-4ac5-ab1f-58116cb5516d), [Natixis – June FX Monitor](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/5e853ab1-4212-4efb-bbd7-4fa9ec3e4bd6).
- External backdrop: Recent BRL softness tracked bouts of USD strength on hawkish Fed signals; consensus noted the real around the low‑5s per USD in early June. Banks with a medium‑term weak‑USD view see scope for BRL to outperform LatAm peers later in 2026 if global rates roll over [FocusEconomics](https://research.bluematrix.com/docs/pdf/9e97d874-a577-44af-8819-ed39505091fc/1e71ae6d-150f-4b35-88dd-cc6cb3416987), [Natixis – May FX Monitor](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/5bf3eb79-06f1-4234-a61c-811e77cf1000).
- Terms of trade: Brazil’s trade surplus stays robust, a structural support for BRL, though commodity‑price volatility remains a swing factor [LBBW FI](https://research.bluematrix.com/docs/pdf/982dd882-1f71-4da4-ae1d-4c275eeab710/aeab69fe-9ba0-48fa-8e92-2a4cb41e7aff).

Outlook
- 1–3 months: Sideways-to-choppy. High carry and solid external balances cushion dips, but upside in the USD on sticky US inflation or risk‑off, plus domestic fiscal/political noise, can lean BRL weaker on spikes. Expect the real to trade broadly in the low‑5s vs USD with volatility around data/events (inflation prints, fiscal news, Fed tone, and local election headlines) [FocusEconomics](https://research.bluematrix.com/docs/pdf/9e97d874-a577-44af-8819-ed39505091fc/1e71ae6d-150f-4b35-88dd-cc6cb3416987), [Natixis – June FX Monitor](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/5e853ab1-4212-4efb-bbd7-4fa9ec3e4bd6).
- 6–12 months: Cautiously constructive. If US disinflation resumes and the dollar eases while COPOM continues a gradual cutting path without re‑igniting inflation expectations—and fiscal credibility holds—banks see room for modest BRL appreciation or at least outperformance versus EM peers, supported by carry and a healthy trade surplus. Key downside risks: renewed USD strength, a re‑acceleration in local inflation that forces policy to stay tighter for longer, and fiscal slippage [Natixis – May FX Monitor](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/5bf3eb79-06f1-4234-a61c-811e77cf1000), [The Economist Intelligence Unit](https://cdn.ravenpack.com/uuid/C61699DC81AB37FC105F8E4710DE382A/filename/eiu_five-year-forecast_2026-06-17_BR.pdf), [LBBW FI](https://research.bluematrix.com/docs/pdf/982dd882-1f71-4da4-ae1d-4c275eeab710/aeab69fe-9ba0-48fa-8e92-2a4cb41e7aff).

What changed after the latest COPOM
- The third consecutive 25 bp cut to 14.25% keeps the easing path intact but signals continued caution given inflation near the top of the target band and lifted forecasts. Markets read it as “carry stays high for longer,” supportive for BRL on calm days but not enough to offset USD spikes or domestic headline risk on its own [US News & World Report](https://money.usnews.com/investing/news/articles/2026-06-17/brazil-central-bank-cuts-rates-for-third-straight-meeting-by-25-bps), [Morningstar](https://www.morningstar.com/news/dow-jones/202606178600/brazils-central-bank-keeps-cutting-despite-hot-prices-update).

Bottom line
- Base case: Range‑bound in the near term with a mild medium‑term appreciation bias, contingent on a softer USD and stable domestic policy signals. Watch US rates/dollar, Brazil’s inflation prints and guidance from COPOM, and any fiscal rule or spending headlines for directional breaks [FocusEconomics](https://research.bluematrix.com/docs/pdf/9e97d874-a577-44af-8819-ed39505091fc/1e71ae6d-150f-4b35-88dd-cc6cb3416987), [Natixis – June FX Monitor](https://research.bluematrix.com/docs/pdf/3de41bc7-bfd2-406d-a320-bef7107a3f1c/5e853ab1-4212-4efb-bbd7-4fa9ec3e4bd6).

---

## 🔟 Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**

| Query Type | Tool Used |
|------------|----------|
| Portfolio positions | `internal_query_database` |
| Internal research | `internal_search_research` |
| Company lookup (fast-mode filters) | `bigdata_lookup_company` |
| Market intelligence (news, filings, transcripts, research, macro/FX) | `bigdata_search` |

**Smart vs. Fast search:**

| Mode | When to use | What you provide |
|------|-------------|------------------|
| **Smart** (default) | Most questions; natural-language and macro/FX topics | Just `query` (entity names, period, content hints); API infers entities, dates, doc types, ranking |
| **Fast** (precision) | Deterministic, filter-driven retrieval | Resolved `entity_ids` / `reporting_entity_ids`, `category`, `document_type`, `days_back`, `rerank_threshold` |

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

---

## 🎯 Next Steps

**Extend this architecture:**

- Add More External Data Sources 
- Implement Distributed Cache for company lookup and Response Caching
   


**Production Considerations:**
- Implement proper error handling for API failures
- Add rate limiting to manage API costs
- Cache frequently requested data
- Monitor performance in LangSmith

- **Advanced Graph Patterns, based on need**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

- **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

- **Context Compression**
   - Strategy to compress the context (i.e. summarizing when context window reaches 80%)

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **Search API Reference**: https://docs.bigdata.com/search-api
- **Knowledge Graph API**: https://docs.bigdata.com/knowledge-graph
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com

## 📊 LangSmith Tracing

![LangSmith Tracing](./static/langsmith_search.png)

---